# Prepare Models and other data for Deployment on Streamlit

Import models from ML workflow and export into app/ dir

## Imports

In [ ]:
import os

os.environ["PYTORCH_MPS_ENABLE"] = "0"
import torch
import joblib
import sys
import numpy as np
from shutil import rmtree
sys.path.append("../")
from src.config import BASE_PATH
from src.data_utils import get_data, get_models

Globals

In [ ]:
##Data
OUTCOME_DICT = {
    "surg": get_data("outcome_surg"),
    "bleed": get_data("outcome_bleed"),
    "asp": get_data("outcome_asp"),
    "mort": get_data("outcome_mort"),
}
MODEL_PREFIX = 'stack'
##Models
model_prefix_list = [MODEL_PREFIX]
model_dir = BASE_PATH / "cal_models"
MODEL_DICT = {}
for outcome in OUTCOME_DICT.keys():
    MODEL_DICT[outcome] = get_models(model_prefix_list, outcome, file_dir=model_dir)

## Models + other data

Models

In [ ]:
model_output_dir = BASE_PATH / "app" / "models"
if model_output_dir.exists():
    rmtree(model_output_dir)
model_output_dir.mkdir(exist_ok=False, parents=True)
for outcome in MODEL_DICT.keys():
    model = MODEL_DICT[outcome][MODEL_PREFIX]
    joblib.dump(model, model_output_dir / f"{outcome}_{MODEL_PREFIX}.joblib")

Thresholds

Probability Distributions

In [ ]:
def get_proba_distributions(y_true, y_proba):
    y_true = np.asarray(y_true)
    pos_scores = y_proba[y_true == 1]
    neg_scores = y_proba[y_true == 0]
    return pos_scores, neg_scores


for outcome_name, outcome_data in OUTCOME_DICT.items():
    

# Save distribution data for percentile feedback
val_pos, val_neg = get_proba_distributions(y_val, y_proba_val)
joblib.dump(
    {"pos": val_pos, "neg": val_neg},
    f"{results_path}/VAL_PROBA_DIST_{outcome_name}.joblib",
)